# Data Curation

The goal of this notebook is to download the data files we'll be using and convert them to formats that can be loaded in subsequent scripts.

We'll download:
- EAGLE-I Data (the target variable)
- NOAA Weather Events
    - NOAA data
    - NWS Zone-County data to help match events with FIPS codes
- ERA5-Land Weather Reanalysis
- County-Level Shapefiles from US Census
- eGrid Subregions
- NRI Hazard Information (county-level)
- Buried Power Line information from Stanford Data Commons

## Imports

In [ ]:
import pandas as pd
import urllib.request
import os
import cdsapi

import dask.dataframe as dd
import xarray as xr
import fastparquet

## EAGLE-I Data

Our target variable comes from the EAGLE-I data set, which provides county-level data for the number of customers without power at a 15-minute cadence.

The full dataset (including data from 2023) appears to only be available via the competition website:
https://thinkonward.com/app/c/challenges/dynamic-rhythms/data

# NOAA Weather Events

The NOAA maintains a database of severe weather events: https://www.ncdc.noaa.gov/stormevents/ftp.jsp

In [ ]:
files_to_download = [
    "StormEvents_details-ftp_v1.0_d2014_c20231116.csv",
    "StormEvents_details-ftp_v1.0_d2015_c20240716.csv",
    "StormEvents_details-ftp_v1.0_d2016_c20220719.csv",
    "StormEvents_details-ftp_v1.0_d2017_c20230317.csv",
    "StormEvents_details-ftp_v1.0_d2018_c20240716.csv",
    "StormEvents_details-ftp_v1.0_d2019_c20240117.csv",
    "StormEvents_details-ftp_v1.0_d2020_c20240620.csv",
    "StormEvents_details-ftp_v1.0_d2021_c20240716.csv",
    "StormEvents_details-ftp_v1.0_d2022_c20241121.csv",
    "StormEvents_details-ftp_v1.0_d2023_c20241216.csv",
    "StormEvents_details-ftp_v1.0_d2024_c20241216.csv"
]

base_url = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"

output_dir = "Data/NOAA_StormEvents"
os.makedirs(output_dir, exist_ok=True)

for file_name in files_to_download:
    file_url = base_url + file_name
    output_path = os.path.join(output_dir, file_name)
    urllib.request.urlretrieve(file_url, output_path)

### NWS Shapefile

We'll need information from the NWS to help match event locations with counties, from https://www.weather.gov/source/gis/Shapefiles/County/bp05mr24.dbx

In [ ]:
url = "https://www.weather.gov/source/gis/Shapefiles/County/bp05mr24.dbx"

output_dir = "Data/NOAA_Cleaned_Data"
os.makedirs(output_dir, exist_ok=True)

output_path = "Data/NOAA_Cleaned_Data/bp05mr24.dbx"
urllib.request.urlretrieve(url, output_path)

# ERA5 data

The European Centre for Medium-Range Weather Forecasts (ECMWF) offers reanalysis datasets for their weather models via the Copernicus Climate Data Store:

https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=download

These files are quite large, so we'll download the data in chunks via their API and then convert to lode-able formats.

### Downloading ERA5 Data

We'll use the API from the Copernicus web store to download the following variables from the ERA5-Land Reanalysis data set:
- Temperature at 2M
- U and V components of wind at 10 M
- Snowfall
- Total Precipitation

Note that these data are sampled at every 9 km. One degree corresponds to approximately 111 km, so the data are sampled at roughly every tenth of a degree. However, this won't be exact, so we'll need to a bit of rounding to be able to merge the ERA5 data with the eaglei/NOAA data.

In [ ]:
output_dir = "Data/ERA5_Data"
os.makedirs(output_dir, exist_ok=True)


c = cdsapi.Client()
for year in range(2014, 2024):
    c.retrieve(
        'reanalysis-era5-land',
        {
            'product_type': 'reanalysis',
            'variable': [
                '10m_u_component_of_wind', '10m_v_component_of_wind', 
                '2m_temperature', 'snowfall', 'total_precipitation'
            ],
            'year': str(year),
            'month': [f'{month:02d}' for month in range(1, 13)],
            'day': [f'{day:02d}' for day in range(1, 32)],
            "time": ["00:00", "06:00", "12:00","18:00"],
            "format": "grib",
            "download_format": "zip",
            "area": [50, -125, 24, -66]
        },
        f'Data/ERA5_Data/ERA5_{year}.grib')

### Converting ERA5 to parquet

To perform the merging of ERA5 and the NOAA-eaglei data, we'll need both to be in parquet (or, at least, non-grib) format

In [ ]:
# Start by converting the ERA5 grib files to parquet
# As part of this process, we'll also need to adjust the index to match the format of the eaglei_noaa index

grib_files = [
    'Data/ERA5_Data/ERA5_2014.grib',
    'Data/ERA5_Data/ERA5_2015.grib',
    'Data/ERA5_Data/ERA5_2016.grib',
    'Data/ERA5_Data/ERA5_2017.grib',
    'Data/ERA5_Data/ERA5_2018.grib',
    'Data/ERA5_Data/ERA5_2019.grib',
    'Data/ERA5_Data/ERA5_2020.grib',
    'Data/ERA5_Data/ERA5_2021.grib',
    'Data/ERA5_Data/ERA5_2022.grib',
    'Data/ERA5_Data/ERA5_2023.grib'
]

for grib_file in grib_files:
    # Load the GRIB file using the cfgrib engine
    grib_data = xr.open_dataset(grib_file, engine='cfgrib', decode_timedelta=True)

    # Convert to an xarray DataFrame
    df = grib_data.to_dataframe()

    # Reset the index to convert the multiindex to columns
    df.reset_index(inplace=True)

    # Remove the time, step, number, and surface variables
    df.drop(columns=['time', 'step', 'number', 'surface'], inplace=True)

    # Rename valid_time as time to facilitate merging with eaglei/NOAA
    df.rename(columns={'valid_time': 'time'}, inplace=True)

    # Round latitude and longitude to the nearest tenth of a degree
    # This is to ensure that the latitude and longitude match the format in the eaglei_noaa index
    df['latitude'] = df['latitude'].round(1)
    df['longitude'] = df['longitude'].round(1)

    # Make the index time, latitude, and longitude
    df.set_index(['time', 'latitude', 'longitude'], inplace=True)

    # Save as Parquet file with the same name but with .parquet extension
    output_file = grib_file.replace('.grib', '.parquet')
    df.to_parquet(output_file)

Ignoring index file '../Data/ERA5_Data/ERA5_2014.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../Data/ERA5_Data/ERA5_2018.grib.5b7b6.idx' older than GRIB file
Ignoring index file '../Data/ERA5_Data/ERA5_2023.grib.5b7b6.idx' older than GRIB file


# Shapefiles

We'll need county-level (technically, FIPS-level) shapefiles to do some of our cleaning and feature engineering

In [ ]:
url = "https://www2.census.gov/geo/tiger/GENZ2023/kml/cb_2023_us_county_500k.zip"
output_path = "Data/County_Shapefiles/cb_2023_us_county_500k.zip"
urllib.request.urlretrieve(url, output_path)

# eGrid Subregions

The EPA has shapefiles to define the various subgrid regions of the electric power grid: https://www.epa.gov/egrid/egrid-mapping-files

Note that there aren't separate shapefiles for each year.

The file naming conventions aren't standardized, so we can't use a loop to download these files

In [ ]:
files_to_download = [
    "https://www.epa.gov/system/files/other-files/2025-01/egrid2023_subregions.zip",
    "https://www.epa.gov/system/files/other-files/2024-05/egrid2022_subregions_shapefile.zip",
    "https://www.epa.gov/system/files/other-files/2023-05/eGRID2021_subregions_shapefile.zip",
    "https://www.epa.gov/system/files/other-files/2022-01/egrid2020_subregions.zip",
    "https://www.epa.gov/sites/default/files/2021-02/egrid2019_subregions.zip",
    "https://www.epa.gov/sites/default/files/2020-03/egrid2018_subregions.zip",
    "https://www.epa.gov/sites/default/files/2020-03/egrid_2016_subregions_shapefiles.zip",
    "https://www.epa.gov/sites/default/files/2017-01/egrid_subregions.zip"
]

output_dir = "Data/eGRID_Data"
os.makedirs(output_dir, exist_ok=True)

for file_name in files_to_download:
    output_path = os.path.join(output_dir, file_name)
    urllib.request.urlretrieve(file_name, output_path)

# NRI

FEMA maintains a database of national risk index variables, including county population and size

Datasets aren't differentiated by filename, so we'll need to manually create a directory for each one

In [ ]:
output_dir = "Data/NRI_Hazard_Info"
os.makedirs(output_dir, exist_ok=True)

# Download the 2023 file
output_dir1 = "Data/NRI_Hazard_Info/NRI_Table_Counties_2023"
os.makedirs(output_dir1, exist_ok=True)
file_name1 = "https://hazards.fema.gov/nri/Content/StaticDocuments/DataDownload//NRI_Table_Counties/NRI_Table_Counties.zip"
output_path1 = os.path.join(output_dir1, file_name1)
urllib.request.urlretrieve(file_name1, output_path1)

# Download the 2021 file
output_dir2 = "Data/NRI_Hazard_Info/NRI_Table_Counties_2021"
os.makedirs(output_dir2, exist_ok=True)
file_name2 = "https://hazards.fema.gov/nri/Content/StaticDocuments/DataDownload/Archive/v118_1/NRI_Table_Counties.zip"
output_path2 = os.path.join(output_dir2, file_name2)
urllib.request.urlretrieve(file_name2, output_path2)

# Download the 2020 file
output_dir3 = "Data/NRI_Hazard_Info/NRI_Table_Counties_2020"
os.makedirs(output_dir3, exist_ok=True)
file_name3 = "https://hazards.fema.gov/nri/Content/StaticDocuments/DataDownload/Archive/v117_0/NRI_Table_Counties.zip"
output_path3 = os.path.join(output_dir3, file_name3)
urllib.request.urlretrieve(file_name3, output_path3)

# Buried Power Lines

The paper "Mapping the Depths: A Stocktake of Underground Power Distribution in United States" (https://arxiv.org/abs/2402.06668) describes researchers' efforts to compute the percent of buried power lines for each US county.

Data are available through te Stanford Data Commons: https://datacommons.stanford.edu/tools/download#pt=County&place=country%2FUSA&sv=distri_ug_rate&dtType=ALL&facets=%7B%7D

It appears that the only way to download the code is through the website. Data should be stored in the directory Data/County_level_Variables/Buried_Power_Lines and the file should be named 'United_States_of_America_County.csv'
